# 🔗 Bloque 3 — Sesión 1: Concepto de correlación, Pearson y Spearman

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **3.1** Concepto de correlación: qué mide y qué no mide. Correlación ≠ causalidad
- **3.2** Correlación de Pearson: coeficiente r, interpretación, limitaciones y visualización matricial
- **3.3** Correlación de Spearman: para variables ordinales o distribuciones no normales

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación de la calidad de los datos basadas en cálculos estadísticos para detectar la cobertura y el sesgo.  
> **(CE c)** Se ha documentado la descripción del resultado de las verificaciones sobre el conjunto de datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 3.1 Concepto de correlación

### ¿Qué mide la correlación?

La **correlación** cuantifica la **fuerza y dirección de la relación lineal** entre dos variables numéricas. Nos dice:
- **Si aumenta una variable, ¿tiende a aumentar o disminuir la otra?** (dirección)
- **¿Con qué regularidad?** (fuerza)

La correlación siempre toma valores entre **−1 y +1**:

| Valor | Interpretación |
|-------|---------------|
| +1 | Relación lineal positiva perfecta |
| +0.7 a +1 | Correlación positiva fuerte |
| +0.3 a +0.7 | Correlación positiva moderada |
| −0.3 a +0.3 | Correlación débil o nula |
| −0.7 a −0.3 | Correlación negativa moderada |
| −1 a −0.7 | Correlación negativa fuerte |
| −1 | Relación lineal negativa perfecta |

In [ ]:
# Galería de correlaciones: distintos valores de r
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flat

escenarios = [
    (+1.0, 'r = +1.0 (lineal perfecta)'),
    (+0.8, 'r = +0.8 (fuerte positiva)'),
    (+0.4, 'r = +0.4 (moderada positiva)'),
    ( 0.0, 'r = 0.0 (sin relación lineal)'),
    (-0.4, 'r = −0.4 (moderada negativa)'),
    (-0.8, 'r = −0.8 (fuerte negativa)'),
    (-1.0, 'r = −1.0 (lineal perfecta)'),
    (None, 'r ≈ 0 pero relación cuadrática'),
]

x = np.linspace(-2, 2, 100)
x_std = (x - x.mean()) / x.std()

def y_con_correlacion(r_obj):
    """Genera y de modo que la correlación muestral con x sea exactamente r_obj."""
    ruido = np.random.normal(0, 1, len(x))
    ruido = ruido - ruido.mean()
    ruido = ruido - (ruido @ x_std) / (x_std @ x_std) * x_std   # ruido incorrelado con x
    ruido = ruido / ruido.std()
    return r_obj * x_std + np.sqrt(1 - r_obj**2) * ruido

for ax, (r_obj, titulo) in zip(axes, escenarios):
    if r_obj is None:
        y = x**2 + np.random.normal(0, 0.15, len(x))
    else:
        y = y_con_correlacion(r_obj)

    r_real, _ = pearsonr(x, y)
    ax.scatter(x, y, s=15, alpha=0.6, color='steelblue')
    ax.set_title(f'{titulo}\n(r calculado = {r_real:+.2f})', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Galería de correlaciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


> 🔎 **Qué observar:** cuanto más cerca está |r| de 1, más se ajusta la nube a una recta. El último panel es el aviso importante: la relación entre las variables es clarísima (una parábola), pero **r ≈ 0** porque no es lineal. Un r cercano a cero significa «no hay relación *lineal*», no «no hay relación».

### Correlación ≠ causalidad

Este es uno de los errores más frecuentes en el análisis de datos. Que dos variables estén correlacionadas **no implica que una cause la otra**. Puede haber:

- **Variable confusora (tercera variable):** una tercera variable causa ambas. *Ejemplo:* el número de salvavidas en una ciudad correlaciona con el número de ahogamientos — ambos dependen del tamaño de la ciudad.
- **Causalidad inversa:** la relación existe pero en sentido contrario al asumido.
- **Correlación espuria:** coincidencia estadística sin relación causal real.

> 💡 **Implicación para ML:** una variable puede ser un buen predictor sin ser causa. Pero si queremos entender el fenómeno o intervenir en él, necesitamos causalidad, no solo correlación.

> ℹ️ **Sobre el p-valor:** en los ejemplos de esta sesión, junto al coeficiente aparece un **p-valor**. Ya lo usamos en los tests de normalidad (Bloque 2, Sesión 2): un p-valor pequeño indica que sería raro obtener una correlación así por puro azar si en realidad no hubiera relación. En la **Sesión 2** de este bloque lo formalizamos para la correlación.

In [ ]:
# Ejemplo clásico: correlación espuria
# Número de películas de Nicolas Cage y ahogamientos en piscinas (datos reales de Tyler Vigen)
anios = np.arange(1999, 2010)
cage_peliculas = [2, 2, 2, 3, 1, 1, 2, 3, 4, 1, 4]
ahogamientos = [109, 102, 102, 98, 85, 95, 96, 98, 123, 94, 102]

r, p = pearsonr(cage_peliculas, ahogamientos)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

color1, color2 = 'steelblue', 'coral'
ax_l = axes[0]
ax_r = ax_l.twinx()
ax_l.bar(anios - 0.2, cage_peliculas, width=0.4, color=color1, alpha=0.8, label='Películas N. Cage')
ax_r.bar(anios + 0.2, ahogamientos, width=0.4, color=color2, alpha=0.8, label='Ahogamientos')
ax_l.set_ylabel('Nº películas', color=color1)
ax_r.set_ylabel('Ahogamientos en piscinas (EE.UU.)', color=color2)
ax_l.set_title(f'Correlación espuria — r = {r:.3f}, p = {p:.3f}')
ax_l.set_xticks(anios)
ax_l.tick_params(axis='x', rotation=45)
h_l, l_l = ax_l.get_legend_handles_labels()
h_r, l_r = ax_r.get_legend_handles_labels()
ax_r.legend(h_l + h_r, l_l + l_r, loc='upper left', fontsize=8)

axes[1].scatter(cage_peliculas, ahogamientos, color='steelblue', s=60)
m, b = np.polyfit(cage_peliculas, ahogamientos, 1)
x_fit = np.linspace(min(cage_peliculas), max(cage_peliculas), 100)
axes[1].plot(x_fit, m*x_fit + b, 'crimson', linewidth=2)
axes[1].set_xlabel('Películas de Nicolas Cage ese año')
axes[1].set_ylabel('Ahogamientos en piscinas')
axes[1].set_title('Scatter: aparente correlación positiva')

plt.tight_layout()
plt.show()
print(f"r = {r:.3f}  |  p = {p:.4f}  — correlación estadísticamente significativa... pero espuria.")

> 🔎 **Qué observar:** r = 0,67 con p < 0,05 pasaría cualquier filtro automático de «correlación significativa», y sin embargo nadie defendería que una cosa causa la otra. Con solo 11 puntos y miles de series posibles para comparar, alguna coincidencia así aparece siempre. La estadística no sustituye al conocimiento del dominio.

---
## 3.2 Correlación de Pearson

La correlación de Pearson mide la **relación lineal** entre dos variables numéricas continuas:

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

**Supuestos:**
- Ambas variables son numéricas continuas
- La relación entre ellas es lineal (o aproximadamente lineal)
- Para **calcular** r no hace falta normalidad. Sí se asume (aproximadamente) para la **inferencia** sobre r: p-valor e intervalos de confianza
- No hay outliers extremos que distorsionen la estimación

**Limitaciones importantes:**
- Solo detecta relaciones **lineales** (puede valer 0 con relación curvilínea perfecta)
- Muy sensible a **outliers**
- No funciona bien con variables ordinales

In [ ]:
# Pearson sobre el dataset Penguins
peng_num = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]

r_matrix = peng_num.corr(method='pearson')
print("Matriz de correlación de Pearson (Penguins):")
print(r_matrix.round(3))

> 🔎 **Qué observar:** la correlación más alta es `flipper_length_mm` – `body_mass_g` (r = 0,87): los pingüinos más grandes tienen aletas más largas. Fíjate también en que `bill_depth_mm` correlaciona **negativamente** con todas las demás. Enseguida veremos que ese signo negativo es engañoso.

In [ ]:
# Heatmap de la matriz de correlación
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(r_matrix, dtype=bool), k=1)  # oculta el triángulo superior (es simétrico)

sns.heatmap(
    r_matrix, mask=mask, annot=True, fmt='.3f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax, square=True
)
ax.set_title('Matriz de correlación de Pearson — Penguins', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix para visualizar todas las relaciones a la vez
g = sns.pairplot(
    penguins, vars=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    hue='species', diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}
)
g.figure.suptitle('Scatter matrix — Penguins (por especie)', y=1.02, fontsize=13)
plt.show()

### ⚠️ Cuidado con los grupos: la paradoja de Simpson

En la matriz global, `bill_length_mm` y `bill_depth_mm` tienen correlación **negativa** (r = −0,23). Pero mira el scatter de ese par en la figura anterior: dentro de cada especie la nube **sube**. Vamos a comprobarlo calculando r por separado para cada especie.

In [ ]:
# Correlación bill_length_mm vs bill_depth_mm: global y por especie
r_global, _ = pearsonr(penguins['bill_length_mm'], penguins['bill_depth_mm'])
print(f"Global (todas las especies): r = {r_global:+.3f}")
for especie, grupo in penguins.groupby('species'):
    r_esp, _ = pearsonr(grupo['bill_length_mm'], grupo['bill_depth_mm'])
    print(f"  {especie:<10} r = {r_esp:+.3f}  (n = {len(grupo)})")

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm',
                hue='species', alpha=0.6, s=25, ax=ax)
for especie, grupo in penguins.groupby('species'):
    sns.regplot(data=grupo, x='bill_length_mm', y='bill_depth_mm',
                scatter=False, ci=None, ax=ax)
sns.regplot(data=penguins, x='bill_length_mm', y='bill_depth_mm',
            scatter=False, ci=None, ax=ax, color='black',
            line_kws={'linestyle': '--', 'linewidth': 1.5})
ax.set_title(f'Recta global (discontinua): r = {r_global:+.2f}; dentro de cada especie la pendiente es positiva')
plt.tight_layout()
plt.show()


> 🔎 **Qué observar:** dentro de cada especie la correlación es **positiva** (entre +0,39 y +0,65), pero al mezclar las tres sale **negativa**. La especie actúa como variable confusora: Gentoo tiene picos largos y poco profundos, Adelie cortos y profundos, y esa diferencia *entre* grupos tapa la relación *dentro* de cada grupo. Es la **paradoja de Simpson**.
>
> 💡 **Implicación para la calidad de datos:** antes de fiarte de una matriz de correlación global, comprueba si el dataset mezcla subpoblaciones (especie, sexo, clase, país…) y repite el cálculo por grupo.

### Efecto de los outliers sobre Pearson

In [ ]:
# Demostración: un solo outlier puede cambiar drásticamente r
np.random.seed(1)
x = np.random.normal(0, 1, 50)
y = 0.1 * x + np.random.normal(0, 1, 50)  # relación casi nula

r_sin, _ = pearsonr(x, y)

# Añadimos un solo outlier influyente
x_out = np.append(x, 5.0)
y_out = np.append(y, 5.0)
r_con, _ = pearsonr(x_out, y_out)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x, y, color='steelblue', s=30, alpha=0.7)
axes[0].set_title(f'Sin outlier\nr = {r_sin:.3f}')
axes[0].set_xlim(-4, 6.5)

axes[1].scatter(x, y, color='steelblue', s=30, alpha=0.7, label='Datos')
axes[1].scatter([5.0], [5.0], color='crimson', s=120, zorder=5, label='Outlier', marker='*')
axes[1].set_title(f'Con 1 outlier\nr = {r_con:.3f}')
axes[1].set_xlim(-4, 6.5)
axes[1].legend()

plt.suptitle('Sensibilidad de Pearson a outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sin outlier: r = {r_sin:.3f}  →  Con outlier: r = {r_con:.3f}")
print(f"Un solo punto de {len(x_out)} multiplica la correlación por {r_con / r_sin:.1f}")

> 🔎 **Qué observar:** 50 puntos sin apenas relación (r ≈ 0,1) pasan a tener una correlación «moderada» (r ≈ 0,45) por un único punto alejado. Por eso conviene mirar siempre el scatter antes de interpretar r, y revisar los outliers (UD1, Bloque 3) antes de calcular correlaciones.

---
## 3.3 Correlación de Spearman

La correlación de Spearman es una alternativa **no paramétrica** a Pearson. En lugar de trabajar con los valores originales, transforma cada valor en su **rango** (posición ordenada) y calcula la correlación de Pearson sobre esos rangos:

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2-1)}$$

donde $d_i$ es la diferencia entre los rangos de $x_i$ e $y_i$.

> ⚠️ Esta fórmula abreviada solo es exacta si **no hay empates** (valores repetidos). Con empates —lo habitual en datos reales— se asignan rangos medios y se calcula Pearson sobre los rangos, que es lo que hacen `scipy.stats.spearmanr` y `df.corr(method='spearman')`.

**Ventajas sobre Pearson:**
- No asume normalidad ni relación lineal: detecta cualquier **relación monótona** (siempre creciente o siempre decreciente)
- Robusta ante outliers (los outliers solo cambian ligeramente su rango)
- Válida para variables **ordinales**

**Cuándo usar Spearman en lugar de Pearson:**
- Distribuciones con sesgo importante o colas pesadas
- Variables ordinales (escala Likert, nivel educativo…)
- Sospecha de relación no lineal pero monótona
- Presencia de outliers que no queremos eliminar

In [ ]:
# Comparación Pearson vs Spearman en distintos escenarios
np.random.seed(42)

escenarios = {
    'Lineal': (lambda x: 0.8*x + np.random.normal(0, 0.5, len(x))),
    'Monótona no lineal (exponencial)': (lambda x: np.exp(1.25*(x + 2)) + np.random.normal(0, 1, len(x))),
    'Con outlier extremo': (lambda x: np.append(0.8*x[:-1] + np.random.normal(0, 0.5, len(x)-1), 15)),
    'No monótona (cuadrática)': (lambda x: x**2 + np.random.normal(0, 0.3, len(x))),
}

x_base = np.linspace(-2, 2, 80)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (titulo, gen_y) in zip(axes.flat, escenarios.items()):
    x = x_base.copy()
    y = gen_y(x)
    if len(y) > len(x): x = np.append(x, 4.0)

    r_p, _ = pearsonr(x, y)
    r_s, _ = spearmanr(x, y)

    ax.scatter(x, y, color='steelblue', s=20, alpha=0.6)
    ax.set_title(f'{titulo}\nPearson r={r_p:.3f}  |  Spearman ρ={r_s:.3f}', fontsize=10)

plt.suptitle('Pearson vs. Spearman en distintos escenarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> 🔎 **Qué observar:**
> - **Lineal:** los dos coeficientes coinciden prácticamente.
> - **Monótona no lineal:** la relación es casi perfecta (siempre creciente), y Spearman lo refleja (ρ cercano a 1); Pearson se queda más bajo porque la nube no es una recta.
> - **Outlier extremo:** Pearson cae con claridad; Spearman apenas se mueve, porque el outlier solo ocupa «el último puesto» del ranking.
> - **No monótona:** ninguno de los dos la detecta. Spearman tampoco es una solución universal.

In [ ]:
# Pearson vs Spearman sobre variables reales: una ordinal y otra muy asimétrica
# pclass = clase del billete (1ª, 2ª, 3ª) → ordinal;  fare = tarifa pagada → muy sesgada
df_ps = titanic[['pclass', 'fare']].dropna()

r_p, p_p = pearsonr(df_ps['pclass'], df_ps['fare'])
r_s, p_s = spearmanr(df_ps['pclass'], df_ps['fare'])

print(f"Pearson   r = {r_p:.4f}  (p = {p_p:.2g})")
print(f"Spearman  ρ = {r_s:.4f}  (p = {p_s:.2g})")
print()
print(f"Sesgo de 'fare': {df_ps['fare'].skew():.2f} → distribución muy asimétrica")
print("'pclass' es ordinal: la distancia entre 1ª y 2ª no tiene por qué ser igual que entre 2ª y 3ª")
print("→ Spearman es más apropiado en este caso")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.stripplot(data=df_ps, x='pclass', y='fare', alpha=0.3, size=3, color='steelblue', ax=axes[0])
axes[0].set_xlabel('Clase (pclass)')
axes[0].set_ylabel('Tarifa (£)')
axes[0].set_title(f'Escala original\nPearson r={r_p:.3f}  |  Spearman ρ={r_s:.3f}')

sns.stripplot(data=df_ps.assign(log_fare=np.log1p(df_ps['fare'])), x='pclass', y='log_fare',
              alpha=0.3, size=3, color='coral', ax=axes[1])
axes[1].set_xlabel('Clase (pclass)')
axes[1].set_ylabel('log(Tarifa + 1)')
axes[1].set_title('Escala log — el escalón entre clases se ve mejor')

plt.tight_layout()
plt.show()


> 🔎 **Qué observar:** los dos coeficientes son negativos (a mayor número de clase, billete más barato), pero Spearman es claramente más fuerte (ρ ≈ −0,69 frente a r ≈ −0,55). Las tarifas extremas de 1ª clase estiran la escala y hacen que Pearson infravalore una relación que, en términos de orden, es muy consistente. En escala logarítmica el escalón entre clases se aprecia mucho mejor.

In [ ]:
# Función: matriz de correlación con Pearson y Spearman comparadas
def comparar_correlaciones(df, columnas):
    """Muestra heatmaps de Pearson y Spearman lado a lado."""
    datos = df[columnas].dropna()
    r_pearson = datos.corr(method='pearson')
    r_spearman = datos.corr(method='spearman')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, matriz, titulo in zip(axes,
                                   [r_pearson, r_spearman],
                                   ['Pearson (lineal)', 'Spearman (monótona)']):
        sns.heatmap(matriz, annot=True, fmt='.3f', cmap='coolwarm',
                    center=0, vmin=-1, vmax=1,
                    linewidths=0.5, linecolor='white',
                    ax=ax, square=True)
        ax.set_title(titulo, fontsize=12)

    plt.suptitle('Comparación Pearson vs. Spearman', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Diferencias notables
    diff = (r_spearman - r_pearson).abs()
    print("Pares con mayor diferencia |Spearman − Pearson|:")
    diff_triu = diff.where(np.triu(np.ones(diff.shape, dtype=bool), k=1))
    top = diff_triu.stack().sort_values(ascending=False).head(5)
    for (c1, c2), d in top.items():
        print(f"  {c1} vs {c2}: Pearson={r_pearson.loc[c1,c2]:.3f}  Spearman={r_spearman.loc[c1,c2]:.3f}  Δ={d:.3f}")

cols_peng = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
comparar_correlaciones(penguins, cols_peng)

> 🔎 **Qué observar:** en Penguins las dos matrices son casi idénticas. Cuando las variables son razonablemente simétricas, sin outliers extremos y con relaciones aproximadamente lineales, Pearson y Spearman cuentan la misma historia. **Una diferencia grande entre ambos es en sí misma una señal de calidad de datos:** avisa de outliers, asimetría fuerte o relaciones curvas. En el Ejercicio 2 verás un caso así.

---
## 🔧 Ejercicios

### Ejercicio 1 — Correlaciones globales y por especie en Penguins

En la sesión hemos visto que el par `bill_length_mm` – `bill_depth_mm` cambia de signo al separar por especie. Comprueba qué ocurre con el resto:
1. Elige una especie y calcula su matriz de correlación de Pearson (solo variables numéricas).
2. Compárala con la matriz global `r_matrix`. ¿Qué pares cambian más? ¿Alguno cambia de signo?
3. El par más correlacionado a nivel global, ¿sigue siéndolo dentro de la especie? ¿Su correlación es mayor o menor? ¿Por qué crees que ocurre?
4. Visualiza ese par con un scatter coloreado por especie.

In [ ]:
# 1. Matriz de Pearson para una especie
# cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
# una_especie = penguins[penguins['species'] == ...]
# ...

# 2. Diferencia con la matriz global
# ...

# 4. Scatter del par más correlacionado, coloreado por especie
# sns.scatterplot(data=penguins, x=..., y=..., hue='species')


### Ejercicio 2 — Pearson vs. Spearman en Diamonds

Aplica `comparar_correlaciones()` al dataset `diamonds` con las variables `price`, `carat`, `x`, `y` y `z` (las tres últimas son las dimensiones del diamante en mm).
1. ¿En qué pares difieren más los dos coeficientes?
2. Revisa el sesgo y los valores máximos y mínimos de las variables implicadas. ¿Ves algún valor imposible o sospechoso?
3. Dibuja el scatter de uno de esos pares. ¿La diferencia se debe a outliers, a una relación curva o a ambas cosas?
4. ¿Qué coeficiente describirías como más fiable aquí y por qué?

In [ ]:
# 1. Comparación de matrices
# comparar_correlaciones(diamonds, [...])

# 2. Sesgo y valores extremos
# diamonds[[...]].skew()
# diamonds[[...]].describe()

# 3. Scatter de un par con mucha diferencia
# ...


### Ejercicio 3 — ¿Causal, confusora o espuria?

Busca en el dataset `titanic` una correlación que parezca sorprendente. Por ejemplo, ¿existe relación entre el número de hermanos/cónyuge a bordo (`sibsp`) y la tarifa pagada (`fare`)? ¿Y entre `parch` y `fare`?

Calcula los coeficientes de Pearson y Spearman, visualiza el scatter y razona: ¿es una relación causal, hay una variable confusora o es una correlación espuria? Pista: averigua qué representa exactamente `fare` (¿precio por persona o por billete?) y qué papel juega `pclass`.

In [ ]:
# df_t = titanic[['sibsp', 'parch', 'fare', 'pclass']].dropna()
# comparar_correlaciones(df_t, ['sibsp', 'parch', 'fare', 'pclass'])

# Scatter sibsp vs fare coloreado por pclass
# sns.scatterplot(data=titanic, x='sibsp', y='fare', hue='pclass', alpha=0.5)

# ¿Qué variable de confusión podría explicar la relación?


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Correlación** | Mide fuerza y dirección de la relación (lineal en Pearson, monótona en Spearman) entre dos variables. Rango [−1, +1]. |
| **Correlación ≠ causalidad** | Correlación alta no implica que una variable cause la otra. |
| **Pearson (r)** | Para relaciones lineales entre variables continuas. Sensible a outliers y a la asimetría. |
| **Paradoja de Simpson** | La correlación global puede tener signo contrario a la de cada subgrupo. Revisa siempre por grupos. |
| **Spearman (ρ)** | Para relaciones monótonas, variables ordinales o distribuciones asimétricas. Robusto. |
| **Heatmap** | Visualización eficiente de la matriz de correlación. Identifica grupos de variables relacionadas. |
| **Scatter matrix** | Muestra todas las relaciones bivariadas a la vez. Útil con 4-8 variables. |
| **Cuándo usar Spearman** | Distribución asimétrica, outliers, variable ordinal o relación no lineal monótona. |
| **Pearson ≠ Spearman** | Una diferencia grande entre ambos avisa de outliers, asimetría o relación curva. |

### 🔜 Próxima sesión
**Bloque 3 — Sesión 2:** Cramér's V para variables categóricas, p-valor e intervalos de confianza, y aplicación a la selección de atributos para el modelado.